In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import time

# 1. 定義 S&P100 的股票代碼列表（共 100 支）
tickers = [
    'AAPL', 'MSFT' , 'AMZN', 'GOOGL', 'GOOG', 'META', 'TSLA', 'BRK-B', 'JPM' , 'JNJ' ,
    'V'   , 'PG'   , 'NVDA', 'HD'   , 'DIS' , 'MA'  , 'UNH' , 'VZ'   , 'INTC', 'KO'  ,
    'PFE' , 'CMCSA', 'NKE' , 'CSCO' , 'WMT' , 'T'   , 'MRK' , 'BA'   , 'XOM' , 'IBM' ,
    'ORCL', 'ACN'  , 'CVX' , 'MDT'  , 'MCD' , 'UPS' , 'HON' , 'ABT'  , 'AMGN', 'COST',
    'LLY' , 'QCOM' , 'AVGO', 'C'    , 'RTX' , 'GILD', 'SBUX', 'CAT'  , 'BLK' , 'USB' ,
    'GS'  , 'AXP'  , 'NOW' , 'TMO'  , 'SPGI', 'ADP' , 'TFC' , 'DE'   , 'CI'  , 'SHW' ,
    'GD'  , 'ADI'  , 'COP' , 'LMT'  , 'AIG' , 'PNC' , 'FDX' , 'APD'  , 'MAR' , 'NEM' ,
    'ECL' , 'PPG'  , 'MET' , 'ETR'  , 'STZ' , 'ZTS' , 'SYK' , 'ICE'  , 'SO'  , 'MCO' ,
    'SPG' , 'TRV'  , 'NUE' , 'DOV'  , 'MTB' , 'EXC' , 'AMP' , 'CTSH' , 'OXY' , 'PEP' ,
    'DG'  , 'EOG'  , 'ADP' , 'ILMN' , 'BMY' , 'VRTX', 'REGN', 'MNST' , 'EXPE', 'WBA'
]

# 2. 定義資料期間
start_date = '2005-01-01'
end_date   = '2015-01-01'

# 3. 下載每日調整收盤價資料
data_daily = yf.download(tickers, start=start_date, end=end_date, auto_adjust=False)['Adj Close']

# 4. 將日資料轉換為月資料：以每月最後一交易日價格為準
monthly_prices = data_daily.resample('ME').last()

# 5. 計算月報酬率（百分比變化）
monthly_returns = monthly_prices.pct_change()

# 6. 將月報酬率資料轉換成長格式（每一行對應一支股票在某個月的資料）
monthly_returns_long = monthly_returns.reset_index().melt(id_vars='Date',
                                                          var_name='公司代碼',
                                                          value_name='月報酬')

# 7. 取得基本面資料（市值及10項資訊）
# 選取的10項資訊因子（注意：將 pegRatio 換成 revenuePerShare）
factors = {
    'ROE': 'returnOnEquity',                 # 股東權益報酬率
    'ROA': 'returnOnAssets',                 # 資產報酬率
    'profitMargins': 'profitMargins',        # 利潤率
    'operatingMargins': 'operatingMargins',  # 營業利潤率
    'trailingPE': 'trailingPE',              # 歷史市盈率
    'forwardPE': 'forwardPE',                # 預期市盈率
    'revenuePerShare': 'revenuePerShare',    # 每股營收
    'debtToEquity': 'debtToEquity',          # 債務對權益比
    'priceToBook': 'priceToBook',            # 市淨率
    'dividendYield': 'dividendYield'         # 股息殖利率
}

fundamental_list = []
for ticker in tickers:
    try:
        tkr = yf.Ticker(ticker)
        info = tkr.info
    except Exception as e:
        print(f"Error fetching info for {ticker}: {e}")
        info = {}
    market_cap = info.get('marketCap', None)
    factor_data = {col: info.get(key, None) for col, key in factors.items()}
    factor_data['公司代碼'] = ticker
    factor_data['市值'] = market_cap
    factor_data['Price'] = monthly_prices[ticker].iloc[-1]
    fundamental_list.append(factor_data)
    time.sleep(1)  # 暫停 1 秒，避免請求過快

fundamental_df = pd.DataFrame(fundamental_list)

# 8. 合併月報酬與基本面資料
final_df = monthly_returns_long.merge(fundamental_df, on='公司代碼', how='left')
final_df.rename(columns={'Date': '月時間'}, inplace=True)
# 調整欄位順序：公司代碼、月時間、月報酬、市值、以及 10 項基本面資訊
final_df = final_df[['公司代碼', '月時間', '月報酬', '市值', 'Price'] + list(factors.keys())]

# 9. 計算月報酬的統計數值：平均、變異數、以及共變異矩陣
# 先用 monthly_returns（DataFrame，索引為日期，欄位為公司代碼）計算
mean_returns = monthly_returns.mean()
variance_returns = monthly_returns.var()
covariance_matrix = monthly_returns.cov()

# 10. 輸出結果到 Excel 檔案，包含多個工作表
output_file = "SP100_data.xlsx"
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    # 工作表1：原始整理資料
    final_df.to_excel(writer, sheet_name='Monthly Data', index=False)

    # 工作表2：每支股票的平均月報酬
    mean_returns_df = mean_returns.reset_index()
    mean_returns_df.columns = ['公司代碼', '平均月報酬']
    mean_returns_df.to_excel(writer, sheet_name='Mean Returns', index=False)

    # 工作表3：每支股票的月報酬變異數
    variance_returns_df = variance_returns.reset_index()
    variance_returns_df.columns = ['公司代碼', '月報酬變異數']
    variance_returns_df.to_excel(writer, sheet_name='Variance', index=False)

    # 工作表4：股票間月報酬共變異矩陣
    covariance_matrix.to_excel(writer, sheet_name='Covariance')

print(f"所有資料已輸出至 {output_file}")


[*********************100%***********************]  99 of 99 completed


所有資料已輸出至 SP100_data.xlsx


In [ ]:
print(data_daily)

Ticker           AAPL        ABT        ACN        ADI        ADP         AIG  \
Date                                                                            
2005-01-03   0.952312  14.011129  18.694983  22.714378  21.654303  806.488403   
2005-01-04   0.962092  13.870040  18.255436  22.096430  21.308624  809.176025   
2005-01-05   0.970518  13.710922  18.184542  22.196302  21.219736  822.611145   
2005-01-06   0.971272  14.020143  18.021477  22.158859  21.076515  823.954651   
2005-01-07   1.041991  14.260318  18.865129  22.190060  21.037022  825.420349   
...               ...        ...        ...        ...        ...         ...   
2014-12-24  24.916599  37.572758  77.415108  45.836311  68.584755   44.143467   
2014-12-26  25.357044  37.696072  77.364258  45.488747  68.552544   44.253273   
2014-12-29  25.339251  37.490540  76.736923  45.327103  68.287041   44.308174   
2014-12-30  25.030046  37.564537  76.457169  44.906807  67.675545   44.323872   
2014-12-31  24.553999  37.01

In [ ]:
print(monthly_prices.AAPL)

Date
2005-01-31     1.157099
2005-02-28     1.349999
2005-03-31     1.254000
2005-04-30     1.085175
2005-05-31     1.196522
                ...    
2014-08-31    22.702652
2014-09-30    22.315050
2014-10-31    23.920851
2014-11-30    26.455956
2014-12-31    24.554007
Freq: ME, Name: AAPL, Length: 120, dtype: float64


In [ ]:
from google.colab import files
files.download("SP100_data.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import time

# 1. 定義 S&P100 的股票代碼列表（共 100 支）
tickers = [
    'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'GOOG', 'META', 'TSLA', 'BRK-B', 'JPM', 'JNJ',
    'V', 'PG', 'NVDA', 'HD', 'DIS', 'MA', 'UNH', 'VZ', 'INTC', 'KO',
    'PFE', 'CMCSA', 'NKE', 'CSCO', 'WMT', 'T', 'MRK', 'BA', 'XOM', 'IBM',
    'ORCL', 'ACN', 'CVX', 'MDT', 'MCD', 'UPS', 'HON', 'ABT', 'AMGN', 'COST',
    'LLY', 'QCOM', 'AVGO', 'C', 'RTX', 'GILD', 'SBUX', 'CAT', 'BLK', 'USB',
    'GS', 'AXP', 'NOW', 'TMO', 'SPGI', 'ADP', 'TFC', 'DE', 'CI', 'SHW',
    'GD', 'ADI', 'COP', 'LMT', 'AIG', 'PNC', 'FDX', 'APD', 'MAR', 'NEM',
    'ECL', 'PPG', 'MET', 'ETR', 'STZ', 'ZTS', 'SYK', 'ICE', 'SO', 'MCO',
    'SPG', 'TRV', 'NUE', 'DOV', 'MTB', 'EXC', 'AMP', 'CTSH', 'OXY', 'PEP',
    'DG', 'EOG', 'ADP', 'ILMN', 'BMY', 'VRTX', 'REGN', 'MNST', 'EXPE', 'WBA'
]

# 2. 定義資料期間
start_date = '2005-01-01'
end_date   = '2015-01-01'

# 3. 下載每日調整收盤價資料（Adj Close）
data_daily = yf.download(tickers, start=start_date, end=end_date, auto_adjust=False)['Adj Close']

# 4. 將日資料轉換為月資料：以每月最後一交易日價格為準
# 若出現 FutureWarning，可使用 'ME'（Month End）
monthly_prices = data_daily.resample('ME').last()

# 5. 計算月報酬率（百分比變化）
monthly_returns = monthly_prices.pct_change()

# 6. 將月報酬率轉換成長格式（每行對應一支股票某個月的數據）
monthly_returns_long = monthly_returns.reset_index().melt(id_vars='Date',
                                                          var_name='公司代碼',
                                                          value_name='月報酬')

# 將月價格資料轉換成長格式（以便取得對應月份的價格）
monthly_prices_long = monthly_prices.reset_index().melt(id_vars='Date',
                                                        var_name='公司代碼',
                                                        value_name='價格')

# 7. 取得基本面資料（市值及10項資訊）
# 選取的10項資訊因子（此處以 revenuePerShare 取代 pegRatio）
factors = {
    'ROE': 'returnOnEquity',                 # 股東權益報酬率
    'ROA': 'returnOnAssets',                 # 資產報酬率
    'profitMargins': 'profitMargins',        # 利潤率
    'operatingMargins': 'operatingMargins',  # 營業利潤率
    'trailingPE': 'trailingPE',              # 歷史市盈率
    'forwardPE': 'forwardPE',                # 預期市盈率
    'revenuePerShare': 'revenuePerShare',    # 每股營收
    'debtToEquity': 'debtToEquity',          # 債務對權益比
    'priceToBook': 'priceToBook',            # 市淨率
    'dividendYield': 'dividendYield'         # 股息殖利率
}

fundamental_list = []
for ticker in tickers:
    try:
        tkr = yf.Ticker(ticker)
        info = tkr.info
    except Exception as e:
        print(f"Error fetching info for {ticker}: {e}")
        info = {}
    market_cap = info.get('marketCap', None)
    factor_data = {col: info.get(key, None) for col, key in factors.items()}
    factor_data['公司代碼'] = ticker
    factor_data['市值'] = market_cap
    # 這裡基本面資訊目前僅取得一次（不隨時間變動）
    fundamental_list.append(factor_data)
    time.sleep(1)  # 暫停 1 秒，降低請求頻率

fundamental_df = pd.DataFrame(fundamental_list)

# 8. 合併月報酬、月價格與基本面資料
# 先合併月報酬與月價格，合併鍵為 Date 及 公司代碼
monthly_data = monthly_returns_long.merge(monthly_prices_long, on=['Date', '公司代碼'], how='left')

# 再合併基本面資料（僅依公司代碼）
final_df = monthly_data.merge(fundamental_df, on='公司代碼', how='left')
final_df.rename(columns={'Date': '月時間'}, inplace=True)

# 調整欄位順序：公司代碼、月時間、月報酬、市值、價格，以及 10 項基本面資訊
final_df = final_df[['公司代碼', '月時間', '月報酬', '市值', '價格'] + list(factors.keys())]

# 9. 計算月報酬的統計數值：平均、變異數、以及共變異矩陣
mean_returns = monthly_returns.mean()
variance_returns = monthly_returns.var()
covariance_matrix = monthly_returns.cov()

mean_returns_df = mean_returns.reset_index()
mean_returns_df.columns = ['公司代碼', '平均月報酬']

variance_returns_df = variance_returns.reset_index()
variance_returns_df.columns = ['公司代碼', '月報酬變異數']

# 10. 輸出結果到 Excel 檔案，包含多個工作表
output_file = "SP100_data.xlsx"
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    # 工作表1：原始整理資料
    final_df.to_excel(writer, sheet_name='Monthly Data', index=False)
    # 工作表2：每支股票的平均月報酬
    mean_returns_df.to_excel(writer, sheet_name='Mean Returns', index=False)
    # 工作表3：每支股票的月報酬變異數
    variance_returns_df.to_excel(writer, sheet_name='Variance', index=False)
    # 工作表4：股票間月報酬共變異矩陣
    covariance_matrix.to_excel(writer, sheet_name='Covariance')

print(f"所有資料已輸出至 {output_file}")

[*********************100%***********************]  99 of 99 completed


所有資料已輸出至 SP100_data.xlsx


In [ ]:
import pandas as pd

# 讀取 Excel 檔案（請確認路徑與檔案名稱）
file_path ='/mnt/data/20250307042402.xlsx'
df = pd.read_excel(file_path)
print(df)

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/20250307042402.xlsx'

In [ ]:
!ls /20250307042402.xlsx

ls: cannot access '/20250307042402.xlsx': No such file or directory
